# Import RDF/BOT Graph into topologic_fast

This notebook demonstrates importing RDF data using the BOT (Building Topology Ontology) into topologic_fast graphs.

## What You'll Learn

1. Parsing RDF/BOT files (Turtle format)
2. Extracting spatial entities and relationships
3. Reconstructing topologic_fast graphs from RDF
4. Visualizing imported building data

## Prerequisites

```bash
pip install rdflib plotly
```

## Note

topologic_fast does not yet have built-in RDF/BOT import. This notebook demonstrates manual parsing that can be integrated once the feature is implemented.

## 1. Setup and Imports

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go

# For RDF parsing
try:
    from rdflib import Graph as RDFGraph, Namespace, URIRef, Literal
    from rdflib.namespace import RDF, RDFS, XSD
    RDF_AVAILABLE = True
    print("rdflib available")
except ImportError:
    RDF_AVAILABLE = False
    print("Warning: Install rdflib for RDF import: pip install rdflib")

## 2. Define RDF Namespaces

In [ ]:
if RDF_AVAILABLE:
    # Define standard namespaces
    BOT = Namespace("https://w3id.org/bot#")
    TOPO = Namespace("http://github.com/wassimj/topologicpy/resources#")
    
    print("Namespaces defined:")
    print(f"  BOT: {BOT}")
    print(f"  TOPO: {TOPO}")
else:
    print("Skipping namespace definition - rdflib not available")

## 3. Create Sample RDF/BOT Data

We'll create a sample RDF file representing a simple building.

In [ ]:
sample_turtle = """
@prefix bot: <https://w3id.org/bot#> .
@prefix topo: <http://github.com/wassimj/topologicpy/resources#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix ex: <http://example.org/building#> .

# Site
ex:Site_001 a bot:Site ;
    rdfs:label "Main Campus" ;
    bot:hasBuilding ex:Building_001 .

# Building
ex:Building_001 a bot:Building ;
    rdfs:label "Office Building" ;
    bot:hasStorey ex:Storey_0 .

# Storey
ex:Storey_0 a bot:Storey ;
    rdfs:label "Ground Floor" ;
    bot:hasSpace ex:Reception, ex:Corridor, ex:Office1, ex:Office2, ex:MeetingRoom, ex:Kitchen .

# Spaces with coordinates
ex:Reception a bot:Space ;
    rdfs:label "Reception" ;
    topo:hasX "2.0"^^xsd:float ;
    topo:hasY "1.5"^^xsd:float ;
    topo:hasZ "1.5"^^xsd:float ;
    bot:adjacentTo ex:Corridor .

ex:Corridor a bot:Space ;
    rdfs:label "Main Corridor" ;
    topo:hasX "5.0"^^xsd:float ;
    topo:hasY "1.5"^^xsd:float ;
    topo:hasZ "1.5"^^xsd:float ;
    bot:adjacentTo ex:Reception, ex:Office1, ex:Office2, ex:MeetingRoom, ex:Kitchen .

ex:Office1 a bot:Space ;
    rdfs:label "Office 1" ;
    topo:hasX "8.0"^^xsd:float ;
    topo:hasY "0.0"^^xsd:float ;
    topo:hasZ "1.5"^^xsd:float ;
    bot:adjacentTo ex:Corridor, ex:Office2 .

ex:Office2 a bot:Space ;
    rdfs:label "Office 2" ;
    topo:hasX "8.0"^^xsd:float ;
    topo:hasY "3.0"^^xsd:float ;
    topo:hasZ "1.5"^^xsd:float ;
    bot:adjacentTo ex:Corridor, ex:Office1, ex:MeetingRoom .

ex:MeetingRoom a bot:Space ;
    rdfs:label "Meeting Room" ;
    topo:hasX "5.0"^^xsd:float ;
    topo:hasY "4.5"^^xsd:float ;
    topo:hasZ "1.5"^^xsd:float ;
    bot:adjacentTo ex:Corridor, ex:Office2, ex:Kitchen .

ex:Kitchen a bot:Space ;
    rdfs:label "Kitchen" ;
    topo:hasX "2.0"^^xsd:float ;
    topo:hasY "4.5"^^xsd:float ;
    topo:hasZ "1.5"^^xsd:float ;
    bot:adjacentTo ex:Corridor, ex:MeetingRoom .
"""

# Save sample data
sample_path = "/tmp/sample_building.ttl"
with open(sample_path, 'w') as f:
    f.write(sample_turtle)

print(f"Sample RDF/BOT saved to: {sample_path}")
print("\nSample content:")
print(sample_turtle[:1000] + "...")

## 4. Parse RDF File

In [ ]:
def parse_bot_file(path):
    """
    Parse a BOT RDF file and extract building information.
    
    Parameters:
    - path: Path to the RDF file
    
    Returns:
    - Dict with sites, buildings, storeys, spaces, and adjacencies
    """
    if not RDF_AVAILABLE:
        return None
    
    # Parse RDF
    g = RDFGraph()
    g.parse(path, format='turtle')
    
    print(f"Parsed {len(g)} triples from {path}")
    
    result = {
        'sites': [],
        'buildings': [],
        'storeys': [],
        'spaces': [],
        'adjacencies': []
    }
    
    # Extract sites
    for site in g.subjects(RDF.type, BOT.Site):
        label = str(g.value(site, RDFS.label, default=str(site)))
        result['sites'].append({'uri': str(site), 'label': label})
    
    # Extract buildings
    for building in g.subjects(RDF.type, BOT.Building):
        label = str(g.value(building, RDFS.label, default=str(building)))
        result['buildings'].append({'uri': str(building), 'label': label})
    
    # Extract storeys
    for storey in g.subjects(RDF.type, BOT.Storey):
        label = str(g.value(storey, RDFS.label, default=str(storey)))
        result['storeys'].append({'uri': str(storey), 'label': label})
    
    # Extract spaces with coordinates
    for space in g.subjects(RDF.type, BOT.Space):
        label = str(g.value(space, RDFS.label, default=str(space)))
        
        # Get coordinates
        x = float(g.value(space, TOPO.hasX, default=Literal(0.0)))
        y = float(g.value(space, TOPO.hasY, default=Literal(0.0)))
        z = float(g.value(space, TOPO.hasZ, default=Literal(0.0)))
        
        result['spaces'].append({
            'uri': str(space),
            'label': label,
            'x': x,
            'y': y,
            'z': z
        })
    
    # Extract adjacencies
    for subj, pred, obj in g.triples((None, BOT.adjacentTo, None)):
        result['adjacencies'].append({
            'source': str(subj),
            'target': str(obj)
        })
    
    return result, g

if RDF_AVAILABLE:
    parsed_data, rdf_graph = parse_bot_file(sample_path)
    
    print("\nParsed data:")
    print(f"  Sites: {len(parsed_data['sites'])}")
    print(f"  Buildings: {len(parsed_data['buildings'])}")
    print(f"  Storeys: {len(parsed_data['storeys'])}")
    print(f"  Spaces: {len(parsed_data['spaces'])}")
    print(f"  Adjacencies: {len(parsed_data['adjacencies'])}")
else:
    print("Skipping parsing - rdflib not available")

## 5. Display Extracted Spaces

In [ ]:
if RDF_AVAILABLE:
    print("Extracted spaces:")
    print("=" * 60)
    
    for space in parsed_data['spaces']:
        print(f"  {space['label']}")
        print(f"    URI: {space['uri']}")
        print(f"    Coordinates: ({space['x']}, {space['y']}, {space['z']})")
        print()
else:
    print("Data not available")

## 6. Convert RDF to topologic_fast Graph

In [ ]:
def bot_to_topologic_graph(parsed_data, include_context=False):
    """
    Convert parsed BOT data to a topologic_fast graph.
    
    Parameters:
    - parsed_data: Dict from parse_bot_file()
    - include_context: If True, include Site, Building, Storey as vertices
    
    Returns:
    - tf.Graph object
    - Dict mapping URIs to vertex indices
    - List of space metadata
    """
    vertices = []
    uri_to_index = {}
    metadata = []
    
    # Add spaces as vertices
    for i, space in enumerate(parsed_data['spaces']):
        v = tf.Vertex.ByCoordinates(space['x'], space['y'], space['z'])
        vertices.append(v)
        uri_to_index[space['uri']] = i
        metadata.append({
            'index': i,
            'uri': space['uri'],
            'label': space['label'],
            'type': 'Space'
        })
    
    # Optionally add context (Site, Building, Storey)
    if include_context:
        # Add storeys at z = -5 (below spaces)
        for storey in parsed_data['storeys']:
            idx = len(vertices)
            v = tf.Vertex.ByCoordinates(5, 2.5, -5)
            vertices.append(v)
            uri_to_index[storey['uri']] = idx
            metadata.append({
                'index': idx,
                'uri': storey['uri'],
                'label': storey['label'],
                'type': 'Storey'
            })
        
        # Add buildings at z = -10
        for building in parsed_data['buildings']:
            idx = len(vertices)
            v = tf.Vertex.ByCoordinates(5, 2.5, -10)
            vertices.append(v)
            uri_to_index[building['uri']] = idx
            metadata.append({
                'index': idx,
                'uri': building['uri'],
                'label': building['label'],
                'type': 'Building'
            })
        
        # Add sites at z = -15
        for site in parsed_data['sites']:
            idx = len(vertices)
            v = tf.Vertex.ByCoordinates(5, 2.5, -15)
            vertices.append(v)
            uri_to_index[site['uri']] = idx
            metadata.append({
                'index': idx,
                'uri': site['uri'],
                'label': site['label'],
                'type': 'Site'
            })
    
    # Create edges from adjacencies
    edges = []
    edge_set = set()  # Avoid duplicate edges
    
    for adj in parsed_data['adjacencies']:
        source_uri = adj['source']
        target_uri = adj['target']
        
        if source_uri in uri_to_index and target_uri in uri_to_index:
            source_idx = uri_to_index[source_uri]
            target_idx = uri_to_index[target_uri]
            
            # Normalize edge key to avoid duplicates
            edge_key = tuple(sorted([source_idx, target_idx]))
            if edge_key not in edge_set:
                edge = tf.Edge.ByStartVertexEndVertex(vertices[source_idx], vertices[target_idx])
                edges.append(edge)
                edge_set.add(edge_key)
    
    # Create graph
    graph = tf.Graph.ByVerticesEdges(vertices, edges)
    
    return graph, uri_to_index, metadata

if RDF_AVAILABLE:
    # Convert to topologic_fast graph
    topo_graph, uri_map, space_metadata = bot_to_topologic_graph(parsed_data, include_context=False)
    
    print("Converted to topologic_fast graph:")
    print(f"  Vertices: {topo_graph.Order()}")
    print(f"  Edges: {topo_graph.Size()}")
    print(f"  Density: {topo_graph.Density():.3f}")
    print(f"  Diameter: {topo_graph.Diameter()}")
    print(f"  Is Bipartite: {topo_graph.IsBipartite()}")
else:
    print("Cannot convert - rdflib not available")

## 7. Analyze the Imported Graph

In [ ]:
if RDF_AVAILABLE:
    print("Graph analysis:")
    print("=" * 60)
    
    vertices = topo_graph.Vertices()
    
    for i, v in enumerate(vertices):
        meta = space_metadata[i]
        degree = topo_graph.VertexDegree(v)
        
        # Find adjacent spaces
        adjacent = topo_graph.AdjacentVertices(v)
        adj_labels = []
        for adj_v in adjacent:
            adj_coords = adj_v.Coordinates()
            for j, orig_v in enumerate(vertices):
                orig_coords = orig_v.Coordinates()
                if (abs(adj_coords[0] - orig_coords[0]) < 0.01 and 
                    abs(adj_coords[1] - orig_coords[1]) < 0.01):
                    adj_labels.append(space_metadata[j]['label'])
                    break
        
        print(f"\n{meta['label']} (degree: {degree})")
        print(f"  Adjacent to: {', '.join(adj_labels) if adj_labels else 'none'}")
else:
    print("Graph not available")

## 8. Visualize the Imported Graph

In [ ]:
def visualize_imported_graph(graph, metadata, title="Imported BOT Graph"):
    """
    Visualize the graph imported from RDF/BOT.
    """
    fig = go.Figure()
    
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Color by room type (based on label)
    color_map = {
        'Reception': '#87CEEB',
        'Corridor': '#D3D3D3',
        'Main Corridor': '#D3D3D3',
        'Office 1': '#90EE90',
        'Office 2': '#90EE90',
        'Meeting Room': '#FFD700',
        'Kitchen': '#FFDAB9',
        'Storey': '#B0C4DE',
        'Building': '#9370DB',
        'Site': '#20B2AA'
    }
    
    # Draw edges
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) >= 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='gray', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        meta = metadata[i]
        color = color_map.get(meta['label'], '#CCCCCC')
        degree = graph.VertexDegree(v)
        
        fig.add_trace(go.Scatter(
            x=[coords[0]],
            y=[coords[1]],
            mode='markers+text',
            marker=dict(
                size=35,
                color=color,
                line=dict(color='black', width=2)
            ),
            text=[meta['label']],
            textposition='middle center',
            textfont=dict(size=8),
            name=meta['label'],
            hovertext=f"{meta['label']}<br>Type: {meta['type']}<br>Degree: {degree}",
            hoverinfo='text'
        ))
    
    fig.update_layout(
        title=title,
        xaxis=dict(title='X', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y'),
        width=800,
        height=600,
        showlegend=False
    )
    
    return fig

if RDF_AVAILABLE:
    fig = visualize_imported_graph(topo_graph, space_metadata, "Office Building from RDF/BOT")
    fig.show()
else:
    print("Cannot visualize - graph not available")

## 9. Query Original RDF with SPARQL

In [ ]:
if RDF_AVAILABLE:
    # Query 1: Find the most connected space
    query1 = """
    PREFIX bot: <https://w3id.org/bot#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT ?space ?label (COUNT(?adjacent) AS ?connections)
    WHERE {
        ?space a bot:Space ;
               rdfs:label ?label ;
               bot:adjacentTo ?adjacent .
    }
    GROUP BY ?space ?label
    ORDER BY DESC(?connections)
    """
    
    print("Query: Most connected spaces")
    print("=" * 40)
    results = rdf_graph.query(query1)
    for row in results:
        print(f"  {row.label}: {row.connections} connections")
else:
    print("Cannot query - rdflib not available")

In [ ]:
if RDF_AVAILABLE:
    # Query 2: Find path between Reception and Kitchen (through adjacencies)
    query2 = """
    PREFIX bot: <https://w3id.org/bot#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT ?start_label ?mid_label ?end_label
    WHERE {
        ?start rdfs:label "Reception" ;
               bot:adjacentTo ?mid .
        ?mid rdfs:label ?mid_label ;
             bot:adjacentTo ?end .
        ?end rdfs:label "Kitchen" .
        ?start rdfs:label ?start_label .
        ?end rdfs:label ?end_label .
    }
    """
    
    print("\nQuery: Path from Reception to Kitchen")
    print("=" * 40)
    results = rdf_graph.query(query2)
    for row in results:
        print(f"  {row.start_label} -> {row.mid_label} -> {row.end_label}")
else:
    print("Cannot query - rdflib not available")

## 10. Use topologic_fast for Pathfinding

In [ ]:
if RDF_AVAILABLE:
    # Find shortest path using topologic_fast
    vertices = topo_graph.Vertices()
    
    # Find Reception and Kitchen vertices
    reception_idx = None
    kitchen_idx = None
    
    for i, meta in enumerate(space_metadata):
        if meta['label'] == 'Reception':
            reception_idx = i
        elif meta['label'] == 'Kitchen':
            kitchen_idx = i
    
    if reception_idx is not None and kitchen_idx is not None:
        v_start = vertices[reception_idx]
        v_end = vertices[kitchen_idx]
        
        # Get distance
        distance = topo_graph.Distance(v_start, v_end)
        print(f"Shortest path from Reception to Kitchen: {distance} steps")
        
        # Get path
        path = topo_graph.Path(v_start, v_end)
        if path:
            path_verts = path.Vertices()
            print(f"\nPath ({len(path_verts)} vertices):")
            
            for pv in path_verts:
                pv_coords = pv.Coordinates()
                # Find matching metadata
                for j, v in enumerate(vertices):
                    v_coords = v.Coordinates()
                    if (abs(pv_coords[0] - v_coords[0]) < 0.01 and 
                        abs(pv_coords[1] - v_coords[1]) < 0.01):
                        print(f"  -> {space_metadata[j]['label']}")
                        break
else:
    print("Graph not available")

## 11. Round-Trip: Export Back to RDF

In [ ]:
def graph_to_bot_rdf(graph, metadata, namespace="http://example.org/exported#"):
    """
    Export a topologic_fast graph back to RDF/BOT.
    """
    if not RDF_AVAILABLE:
        return None
    
    g = RDFGraph()
    NS = Namespace(namespace)
    
    g.bind("bot", BOT)
    g.bind("topo", TOPO)
    g.bind("ex", NS)
    
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Create spaces
    vertex_uris = {}
    for i, v in enumerate(vertices):
        meta = metadata[i]
        coords = v.Coordinates()
        
        # Create URI from label
        safe_label = meta['label'].replace(' ', '_')
        space_uri = NS[safe_label]
        vertex_uris[i] = space_uri
        
        g.add((space_uri, RDF.type, BOT.Space))
        g.add((space_uri, RDFS.label, Literal(meta['label'])))
        g.add((space_uri, TOPO.hasX, Literal(coords[0], datatype=XSD.float)))
        g.add((space_uri, TOPO.hasY, Literal(coords[1], datatype=XSD.float)))
        g.add((space_uri, TOPO.hasZ, Literal(coords[2], datatype=XSD.float)))
    
    # Create adjacencies from edges
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) >= 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            
            source_idx = None
            target_idx = None
            
            for i, v in enumerate(vertices):
                v_coords = v.Coordinates()
                if abs(p1[0] - v_coords[0]) < 0.01 and abs(p1[1] - v_coords[1]) < 0.01:
                    source_idx = i
                if abs(p2[0] - v_coords[0]) < 0.01 and abs(p2[1] - v_coords[1]) < 0.01:
                    target_idx = i
            
            if source_idx is not None and target_idx is not None:
                g.add((vertex_uris[source_idx], BOT.adjacentTo, vertex_uris[target_idx]))
                g.add((vertex_uris[target_idx], BOT.adjacentTo, vertex_uris[source_idx]))
    
    return g

if RDF_AVAILABLE:
    # Export back to RDF
    exported_rdf = graph_to_bot_rdf(topo_graph, space_metadata)
    
    print(f"Exported {len(exported_rdf)} triples")
    print("\nExported RDF (Turtle):")
    print("=" * 60)
    print(exported_rdf.serialize(format='turtle'))
else:
    print("Cannot export - rdflib not available")

## Summary

In this notebook, we demonstrated:

1. **Parsing RDF/BOT files** using rdflib
2. **Extracting spatial entities** (sites, buildings, storeys, spaces)
3. **Converting to topologic_fast graphs** from adjacency relationships
4. **Analyzing connectivity** using graph methods
5. **Querying with SPARQL** for spatial relationships
6. **Pathfinding** using topologic_fast graph algorithms
7. **Round-trip export** back to RDF/BOT

### Key Functions

- `parse_bot_file()` - Parse RDF and extract BOT entities
- `bot_to_topologic_graph()` - Convert to tf.Graph
- `graph_to_bot_rdf()` - Export back to RDF

### Limitations

- **No built-in import** - topologic_fast doesn't yet have Graph.ByBOTPath()
  - NOTE: This feature is not yet implemented in topologic_fast
- **No geometry reconstruction** - We import coordinates but not full 3D geometry
  - NOTE: BREP import is not yet implemented in topologic_fast
- **Manual metadata handling** - Without Dictionary, metadata is stored separately

### Use Cases

- Importing building data from BIM tools
- Integrating with semantic web applications
- Converting between graph formats
- Spatial analysis of linked building data